In [ ]:
print("start")

In [ ]:
!pip install torch torchvision lightning lovely-tensors

In [ ]:
%%writefile VAE_testing_regnet.py
import torch
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim.lr_scheduler import StepLR
import torch.optim as optim
import os
from PIL import Image
import pickle
import pytorch_lightning as pl
import timm
import wandb
from pytorch_lightning.loggers import WandbLogger
from datetime import datetime
# from model_MobileFaceNet import MobileFacenet
# import lpips
import itertools
import lovely_tensors as lt
from pytorch_lightning.callbacks import ModelCheckpoint

lt.monkey_patch()


device = "cuda" if torch.cuda.is_available() else "cpu"
data_path = "/kaggle/input/ffhq-128-70k"

from logging import setLogRecordFactory

class FlatImageDataset(Dataset):
    def __init__(self, paths=[], transform=None):
        self.paths = paths
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img) if self.transform else img


    @staticmethod
    def init_datasets(root=data_path, train_ratio = 0.7, train_transform=None, val_transform=None):
        paths = [os.path.join(root, f) for f in os.listdir(root)]
        train_paths, val_paths = random_split(paths, [train_ratio, 1-train_ratio])
        return FlatImageDataset(paths=train_paths, transform=train_transform), FlatImageDataset(paths=val_paths, transform=val_transform)



class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super(ConvBlock, self).__init__()
        # sequential block consisting of a 2d convolution,
        # batch normalization, and leaky relu activation
        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,  # number of input channels
                out_channels=out_channels,  # number of output channels
                kernel_size=3,  # size of the convolutional kernel
                stride=2,  # stride of the convolution
                padding=1,  # padding added to the input
            ),
            nn.BatchNorm2d(out_channels),  # normalize the activations of the layer
            nn.LeakyReLU(),  # apply leaky relu activation
        )
    def forward(self, x):
        # pass the input through the sequential block
        return self.block(x)


class ConvTBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super(ConvTBlock, self).__init__()
        # sequential block consisting of a transposed 2d convolution,
        # batch normalization, and leaky relu activation
        self.block = nn.Sequential(
            nn.ConvTranspose2d(
                in_channels,  # number of input channels
                out_channels,  # number of output channels
                kernel_size=3,  # size of the convolutional kernel
                stride=2,  # stride of the convolution
                padding=1,  # padding added to the input
                output_padding=1,  # additional padding added to the output
            ),
            # nn.BatchNorm2d(out_channels),  # normalize the activations of the layer
            nn.ReLU(),  # apply leaky relu activation
        )
    def forward(self, x):
        return self.block(x)



class InceptionBlock(nn.Module):
    def __init__(self, in_channels, f1, f3_r, f3, f5_r, f5, f_pool):
        """
        Args:
            in_channels: Number of input channels.
            f1: Number of filters in 1x1 conv branch.
            f3_r: Number of filters in 1x1 conv before 3x3 conv.
            f3: Number of filters in 3x3 conv.
            f5_r: Number of filters in 1x1 conv before 5x5 conv.
            f5: Number of filters in 5x5 conv.
            f_pool: Number of filters in the pooling branch.
        """
        super(InceptionBlock, self).__init__()
        
        # 1x1 conv branch
        self.branch1 = nn.Conv2d(in_channels, f1, kernel_size=1, stride=1, padding=0)
        
        # 1x1 -> 3x3 conv branch
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, f3_r, kernel_size=1, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv2d(f3_r, f3, kernel_size=3, stride=1, padding=1)
        )
        
        # 1x1 -> 5x5 conv branch
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, f5_r, kernel_size=1, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv2d(f5_r, f5, kernel_size=5, stride=1, padding=2)
        )
        
        # Pooling branch
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, f_pool, kernel_size=1, stride=1, padding=0)
        )
        
        self.relu = nn.ReLU()

    def forward(self, x):
        b1 = self.relu(self.branch1(x))
        b2 = self.relu(self.branch2(x))
        b3 = self.relu(self.branch3(x))
        b4 = self.relu(self.branch4(x))
        
        # Concatenate along channel dimension
        return torch.cat([b1, b2, b3, b4], dim=1)

    def __str__(self):
        return "Inception"


class Inception_Decoder(nn.Module): # BL = Bottle Neck
    def __init__(self, convs=[3, 12, 20, 24, 28, 32], act_fn = nn.ReLU(), out_act=nn.Tanh()):
        super().__init__()
        # act_fn = nn.ReLU()
        self.layers = nn.Sequential()
        self.layers.append(nn.BatchNorm2d(convs[-1][0]))
        for i in range(len(convs)-1, 0, -1):
            in_channels = convs[i][0]
            out_channels = convs[i-1][0]

            filters = convs[i][1]

            
            
            # print(convs[i])
            self.layers.append(
                    nn.ConvTranspose2d(in_channels, in_channels, kernel_size=3, stride=2, padding=1, output_padding=1))
    
            self.layers.append(act_fn)
    
            if i == 1:
                self.layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1))
                self.layers.append(act_fn)
                self.layers.append(
                    nn.ConvTranspose2d(3, 3, kernel_size=3, stride=2, padding=1, output_padding=1))
                self.layers.append(act_fn)
                self.layers.append(nn.Conv2d(3, 3, kernel_size=3, stride=1, padding='same'))

            # self.layers.append(convBlock(in_channels, out_channels))
            else:    
                self.layers.append(
                        InceptionBlock(in_channels, filters["f1"], filters['f3_r'], filters["f3"], filters["f5_r"], filters["f5"], filters["f_pool"]))
    
            # if i > 1:
                self.layers.append(act_fn)
    
          # self.layers.append(nn.BatchNorm2d(convs[i - 1]))

        

        
        # output handling for testing multiple output activation functtions
        self.out_act = out_act
        self.input_normalize = nn.Identity()

    def forward(self, x):
        x = self.layers(x)
        
        x = self.out_act(x)
        
        return x


class face_VAE(pl.LightningModule):
    def __init__(self, in_channels, latent_dim=512, hidden_dims = None, trans=None, learning_rate=0.001) -> None:
        super().__init__()
        self.learning_rate = learning_rate

        self.latent_dim = latent_dim  # dimensionality of the latent space

        # build the encoder using regnet decoder
        self.encoder = timm.create_model("regnety_004.pycls_in1k", pretrained=False)

        self.latentConv = nn.Conv2d(
               440,  # number of input channels
               out_channels=440,  # number of output channels
               kernel_size=2,  # size of the convolutional kernel
               stride=2,  # stride of the convolution
               padding=0,  # padding added to the input
            )

        self.fc_mu = nn.Linear(hidden_dims[-1] * 4, latent_dim) # *4 cuz it is 2x2 spatial dim
        # fully connected layer for the variance of the latent space
        self.fc_var = nn.Linear(hidden_dims[-1] * 4, latent_dim)# *4 cuz it is 2x2 spatial dim
        # build the decoder using transposed convolutional blocks
        # fully connected layer to expand the latent space
        self.decoder_input = nn.Linear(latent_dim, hidden_dims[-1] * 4)
        hidden_dims.reverse()  # reverse the hidden dimensions for the decoder

        convs = [
            (3, None),
            (40, {"f1": 3, "f3_r": 10, "f3": 3, "f5_r": 10, "f5": 5, "f_pool": 3}),
            (140, {"f1": 10, "f3_r": 35, "f3": 15, "f5_r": 35, "f5": 10, "f_pool": 5}),
            (240, {"f1": 35, "f3_r": 60, "f3": 45, "f5_r": 60, "f5": 35, "f_pool": 25}),
            (350, {"f1": 60, "f3_r": 90, "f3": 80, "f5_r": 90, "f5": 60, "f_pool": 40}),
            (440, {"f1": 70, "f3_r": 110, "f3": 130, "f5_r": 110, "f5": 80, "f_pool": 70})
        ]
        
        self.decoder = Inception_Decoder(convs, out_act=nn.Identity())


        self.beta = 0.0
        self.reconstruction_files = [f"{data_path}/1.png", f"{data_path}/10004.png", f"{data_path}/10010.png", f"{data_path}/10012.png", f"{data_path}/10053.png", f"{data_path}/10077.png"]
        transform=trans

        self.reconstruction_images = [
            transform(Image.open(img).convert('RGB')).unsqueeze(0) for img in self.reconstruction_files
        ]

        self.hidden_dims = hidden_dims

    def encode(self, input):
        # pass the input through the encoder
        result = self.encoder.forward_features(input) # 440x4x4
        result = self.latentConv(result) # 440x2x2

        # flatten the result for the fully connected layers
        result = torch.flatten(result, start_dim=1)
        # compute the mean of the latent space

        mu = self.fc_mu(result)
        # compute the log variance of the latent space
        log_var = self.fc_var(result)
        return mu, log_var


    def decode(self, z):
        # expand the latent space
        result = self.decoder_input(z)
        # reshape the result for the transposed convolutions
        result = result.view(-1, self.hidden_dims[0], 2, 2)
        # pass the result through the decoder
        result = self.decoder(result)
        return result

    def reparameterize(self, mu, logvar):
        # compute the standard deviation from the log variance
        std = torch.exp(0.5 * logvar)
        # sample random noise
        eps = torch.randn_like(std)
        # compute the sample from the latent space
        return eps * std + mu

    def forward(self, input):
        # encode the input to the latent space
        mu, log_var = self.encode(input)
        # sample from the latent space
        z = self.reparameterize(mu, log_var)
        # decode the sample, and return the reconstruction
        # along with the original input, mean, and log variance
        return self.decode(z), mu, log_var

    def loss_fn(self, recon, true, mu, logvar):
        # lpips_loss = torch.abs(self.loss_fn_vgg(true, recon)).mean()
        mse_loss = F.mse_loss(recon, true)
        beta = self.beta
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

        return mse_loss + beta*kl_loss, mse_loss, beta*kl_loss


    def training_step(self, batch, batch_idx):
        data = batch.to(self.device).float()
        x_recon, mu, logvar = self(data)
        loss, mse_loss, kl_loss = self.loss_fn(x_recon, data, mu, logvar)

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("train_MSE_epoch", mse_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_klD_epoch", kl_loss, on_step=False, on_epoch=True, prog_bar=True)



        if batch_idx % 400 == 0:
            self.log_reconstructions()

        return loss

    def validation_step(self, batch, batch_idx):
        data = batch.to(self.device).float()
        x_recon, mu, logvar = self(data)
        loss, mse_loss, kl_loss = self.loss_fn(x_recon, data, mu, logvar)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_MSE", mse_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_klD", kl_loss, on_step=False, on_epoch=True, prog_bar=True)

        return loss

    def on_train_epoch_end (self):
        # Call log_reconstructions every 20 epochs
        # if (self.current_epoch + 1) % 1 == 0:  # +1 because current_epoch is zero-indexed
        self.log_reconstructions()

        # if (self.current_epoch % 20 == 0) and (self.current_epoch > 20):
        if (self.current_epoch % 20 == 0) and (self.current_epoch > 0):
            self.beta = min(self.beta + 0.00002, 0.00025)
        



    def log_reconstructions(self):
        self.eval()
        with torch.no_grad():
            logs = {}
            reconstructions = []
            for idx, img_tensor in enumerate(self.reconstruction_images):
                img_tensor = img_tensor.to(self.device)
                reconstructed, mu, logvar = self(img_tensor)
                reconstructed = self.unnormalize(reconstructed.cpu()[0]) # [0] for first batch

                rec_np = (reconstructed * 255).clamp(0, 255).byte().numpy().transpose(1, 2, 0).astype(np.uint8)

                logs[f"recon_{idx}"] = wandb.Image(rec_np, caption=f"Epoch {self.current_epoch}")

                filename = f"recon_epoch_{self.current_epoch}_idx{idx}.png"
                rec_img = Image.fromarray(rec_np)
                rec_img.save(filename)


            if self.logger is not None:
                self.logger.experiment.log(logs)


    def unnormalize(self, img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
        """
        Unnormalizes a tensor image that was normalized with the given mean and std.
        Args:
            img (Tensor): Image tensor of shape (C, H, W)
            mean (list): Mean values used for normalization
            std (list): Std values used for normalization
        Returns:
            Tensor: Unnormalized image tensor.
        """
        # Clone to avoid modifying the original tensor
        unnorm = img.clone()
        for t, m, s in zip(unnorm, mean, std):
            t.mul_(s).add_(m)

        return torch.clamp(unnorm, 0, 1)



    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate)

        scheduler = {
        'scheduler': StepLR(optimizer, step_size=5, gamma=0.99, verbose=True),
        'interval': 'epoch',  # Adjust learning rate at the end of each epoch
        'frequency': 1,       # Apply the scheduler every epoch
        }
        return [optimizer], [scheduler]

    def on_epoch_end(self):
        if self.current_epoch < 80:
            for param_group in self.trainer.optimizers[0].param_groups:
                param_group['lr'] = self.learning_rate # Keeps LR constant until epoch 50


train_transform = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(degrees=(0,15)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
    ])



print("loading dataset")
train_dataset, val_dataset = FlatImageDataset.init_datasets(train_transform=train_transform, val_transform=val_transform)

print("done")

times = 1 # when using good gpu -> larger batch size -> larger learning rate (linearly with batch size) -> multiply these with times
batch_size = 128 * times
learning_rate = 0.001 * (times/1)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False, num_workers=3)



model = face_VAE(in_channels=3, latent_dim=512, hidden_dims=[32, 64, 128, 256, 360, 440], trans=val_transform, learning_rate=0.0002)
# model.load_state_dict(torch.load("/kaggle/input/test19/pytorch/default/2/model_test 19 latent512 regnet2.pth").state_dict())

wandb.login(key="urkey")
exp_name = "test 20 latent512 regnet+inception"
wandb_logger = WandbLogger(project="Human AE", name=exp_name,
                               group="test VAE",
                               resume="allow")

config = {
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "model": "regnety16AE",
    "epochs": 20,
    "conv_layers": [3, 64, 111, 222, 444, 888],
    "out_act": "identity"
    # Add other hyperparameters or configurations here
}

checkpoint_callback = ModelCheckpoint(
    dirpath='checkpoints/',      # directory where checkpoints will be saved
    filename='model-{epoch:02d}', # checkpoint filename pattern
    every_n_epochs=21,           # save a checkpoint every 30 epochs
    save_top_k=-1                # save all checkpoints (not only the best ones)
)

trainer = pl.Trainer(
        max_epochs=100,  # Adjust as needed
        callbacks=[checkpoint_callback],
        devices='auto',
        precision='16-mixed',
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        logger=wandb_logger,
        log_every_n_steps=1
    )

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
torch.save(model, f"model_{exp_name}.pth")


wandb.finish()
print("finish")

In [ ]:
!python VAE_testing_regnet.py